# 6 — Full GPT module

**Before:** notebook **5** (one block).

**This notebook:** stack blocks into `GPT`, forward pass + loss.


In [ ]:
# --- Setup: find repo root (llm-c-from-scratch or Cursor workbook) ---
import sys
from pathlib import Path


def find_llm_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "llmc" / "__init__.py").is_file():
            return base
        nested = base / "llm-c-from-scratch"
        if (nested / "llmc" / "__init__.py").is_file():
            return nested
    return Path.cwd()


ROOT = find_llm_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from llmc.notebook_utils import data_path, checkpoint_path

DATA = data_path(ROOT)
CHECKPOINT = checkpoint_path(ROOT)
print("ROOT", ROOT.resolve())
print("data", "OK" if DATA.is_file() else "missing")


In [ ]:
import torch
from llmc.data import CharTokenizer, load_text
from llmc.model import GPT, GPTConfig

text = load_text(DATA)
tokenizer = CharTokenizer.from_text(text)
config = GPTConfig.tiny(vocab_size=tokenizer.vocab_size, block_size=64)
model = GPT(config)

print(model)
print(f"Parameters: {model.count_parameters():,}")


In [ ]:
x = torch.randint(0, tokenizer.vocab_size, (4, 32))
logits, loss = model(x, x)
print("logits:", logits.shape)
print("loss (random targets):", loss.item())


In [ ]:
# --- Fast verify (seconds) ---
import subprocess

tests = ROOT / "tests" / "test_model.py"
if tests.is_file():
    cmd = [sys.executable, "-m", "pytest", str(tests), "-q", "-k", "gpt_forward or generate"]
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=str(ROOT), check=True)
    print("pytest OK")
else:
    print("Skip pytest:", tests)
